<a href="https://colab.research.google.com/github/snigdhanigam-cpu/Bicycle-routing-/blob/main/Assignment_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Submission instruction

1. Make a copy of this notebook
2. Complete the tasks
3. Make the copy editable by instructor (shi.feng@gwu.edu) and grader (justin.mittereder@gwmail.gwu.edu).
4. Submit through [this form](https://forms.gle/P1GWGWRvJTsTvEx9A) and make sure to pick "Assignment 3".

# Assignment 3 - A gradient-descent-trained character-level language model

In this assignment, you will complete the training of a character language model for generating names.

## Step 1: data preparation

In the example we gave in class, we used two special characters to indicate the beginning and the end of the sequence (a name). We actually do not need two special characters---one would be sufficient. In this example, we use the same special character for both the beginning and the end of names.

In [ ]:
!wget https://raw.githubusercontent.com/karpathy/makemore/refs/heads/master/names.txt

In [ ]:
import torch
import torch.nn.functional as F

# ── Data loading ──
words = open("names.txt", "r").read().splitlines()

# ── Vocabulary ──
SENTINEL = "."  # single token for both start and end
chars = sorted(set("".join(words)))
stoi = {c: i + 1 for i, c in enumerate(chars)}
stoi[SENTINEL] = 0
itos = {i: c for c, i in stoi.items()}
VOCAB_SIZE = len(stoi)

# ── Build training pairs ──
def build_bigrams(words, stoi):
    xs, ys = [], []
    for w in words:
        chs = [SENTINEL] + list(w) + [SENTINEL]
        for c1, c2 in zip(chs, chs[1:]):
            xs.append(stoi[c1])
            ys.append(stoi[c2])
    return torch.tensor(xs), torch.tensor(ys)

xs, ys = build_bigrams(words, stoi)

## Step 2: Model construction

Our model consists of a single matrix of parameters. In the example given in class, we directly computed the value for these parameters, and each entry `P[ix1, ix2]` computed $P[ix2 | ix1]$.

Here, instead of manually calculating these values, we will use gradient descent to learn the parameters. We will initialize the matrix `W` randomly, and treat language modeling as a classification problem.

## Your task: complete the forward pass

Remeber that this language model is just like a multi-class classificationo model we have seen before. Each example contains an input (previous character), and a label (current character), and we use softmax to make sure our classification model produces a valid probability distribution over the vocabulary (26 characters + sentinel).

See how the `sample_name` function is implemented, then implement the forward pass. Note that this `get_probs` function will be called in the training loop (implemented in the next cell) like so:

    probs = get_probs(W, xs)
    loss = -probs[torch.arange(len(ys)), ys].log().mean()
    loss.backward()

So get_probs should return the model's prediction (output distribution over vocabulary) for all examples in xs. You can use `F.one_hot` to make your implementation easier, but it's not required.

In [ ]:
# ── Model: a single learnable weight matrix ──
W = torch.randn((VOCAB_SIZE, VOCAB_SIZE), requires_grad=True)

# ── Sampling ──
def sample_name(W, stoi, itos, max_len=30):
    """Generate one name by repeatedly sampling from the bigram model."""
    idx = stoi[SENTINEL]
    chars = []
    for _ in range(max_len):
        logits = W[idx]                       # row of W for current char
        p = F.softmax(logits, dim=0)
        idx = torch.multinomial(p, num_samples=1).item()
        if idx == stoi[SENTINEL]:
            break
        chars.append(itos[idx])
    return "".join(chars)

def get_probs(W, xs):
    """Forward pass: one-hot encode → linear → softmax."""

    """
    TODO: Implement the forward pass
    For context, this function will be called in the training loop (implemented in the next cell) like so:

    probs = get_probs(W, xs)
    loss = -probs[torch.arange(len(ys)), ys].log().mean()
    loss.backward()

    So get_probs should return the model's prediction (output distribution over vocabulary) for all examples in xs.
    """


## Step 3: train the model

You should observe that the loss continues to decrease, and at the end, the model can generate reasonable looking names.

In [ ]:
# ── Training ──
optimizer = torch.optim.SGD([W], lr=50.0)

for epoch in range(500):
    optimizer.zero_grad()
    probs = get_probs(W, xs)
    loss = -probs[torch.arange(len(ys)), ys].log().mean()
    loss.backward()
    optimizer.step()
    if epoch % 20 == 0 or epoch == 99:
        print(f"epoch {epoch:3d}  loss {loss.item():.4f}")


print("\n— Sampled names —")
for _ in range(10):
    print(sample_name(W, stoi, itos))